# Prototyping Visualization Tools

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np 
from pygltflib import GLTF2

from src.viz import plot_static_quat
from src.viz import plot_static_skeleton
from src.viz import DEFAULT_ROTATION_PLOT_LAYOUT, DEFAULT_SKELETON_PLOT_LAYOUT
from src.utils import parse_clip, extract_quats, extract_frame
from src.utils import glb_get_name_hierarchy
from src.kinematics import forward_kinematics_tree

---
## Static Rotation Visualization

In [3]:
axis  = np.array([1, 0, 0])
angle = 3 * np.pi / 4
quat1 = np.concatenate([
    np.sin(angle / 2) * axis / np.linalg.norm(axis), 
    [np.cos(angle / 2)]
])

axis  = np.array([1, 0, 2])
angle = np.pi / 2
quat2 = np.concatenate([
    np.sin(angle / 2) * axis / np.linalg.norm(axis), 
    [np.cos(angle / 2)]
])

axis  = np.array([1, -1, -1])
angle = np.pi
quat3 = np.concatenate([
    np.sin(angle / 2) * axis / np.linalg.norm(axis), 
    [np.cos(angle / 2)]
])

layout = DEFAULT_ROTATION_PLOT_LAYOUT
layout["title"] = "Static Quaternion Visualizer"
plot_static_quat([quat3], colors=["red", "green", "blue"], reference=[0, 0, 1], layout_options=layout)

---
## Static Skeleton

In [4]:
model_path = "model/Alex_Rig_v2.4_rokoko_wface_nov30.glb"
gltf = GLTF2().load(model_path)
bind_model, joint_names = glb_get_name_hierarchy("rootx", gltf.nodes)

In [8]:
iden_quats = dict((name, [[0,0,0,1]]) for name in joint_names)
anim_pos = forward_kinematics_tree(bind_model, iden_quats)
frame_pos = extract_frame(anim_pos, 0)

layout = DEFAULT_SKELETON_PLOT_LAYOUT
layout["title"] = "Bind T-Pose (?)"
plot_static_skeleton(frame_pos, layout_options=layout)

In [9]:
data = np.load("data/npz/clip_72.npz", allow_pickle=True)
clip = data["clip"].item()
clip = parse_clip(clip)

anim = clip["animationClip"]
clip_name = anim["name"]
tracks = anim["tracks"]

In [11]:
frame_idx = 200
clip_quats = extract_quats(clip)
clip_frame_quats = extract_frame(clip_quats, frame_idx)
clip_frame_pos = forward_kinematics_tree(bind_model, clip_frame_quats)

layout = DEFAULT_SKELETON_PLOT_LAYOUT
layout["title"] = f"Frame #{frame_idx} from {clip_name}"
layout["template"] = "simple_white"
plot_static_skeleton(clip_frame_pos, layout_options=layout)